# Single-dong analysis: intra-dong mode split, dong by dong

`build_od_mode_share.py` builds the city-wide tensor $S[i,j,k]$ and prints **one** intra-dong (diagonal) mode split for all of Seoul.
Here we do the same thing **separately for each dong** $d$, using only the trips that start and end inside $d$ ($i=j=d$):

$$
T[d,k] = \text{trips}(d \to d \text{ by mode } k), \qquad
P[d,k] = \frac{T[d,k]}{\sum_{k'} T[d,k']} = S[d,d,k]
$$

For each dong we also keep the trip-weighted mean distance and time of its intra-dong trips per mode.
We then ask whether these per-dong statistics differ, and how:

1. **How large is the spread?** Distribution of $P[d,k]$ over dongs, compared with the pooled city-wide split.
2. **Is the spread real?** Compare it with what multinomial sampling noise alone would give, and with each dong's own weekday/weekend variation.
3. **What does it follow?** Gu membership, dong size, volume, self-containment, and clusters of dongs with similar mode profiles (mapped).
4. **Intra vs outgoing.** Does a dong's intra-dong split look like the split of the trips leaving it?

Same conventions as the script: dongs in `nodes_dongs.csv` row order, modes 4–9 by default (shares re-normalised over the kept modes), weekly weight by default.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage

try:
    import geopandas as gpd
except ImportError:
    gpd = None

# ---- configuration (same meaning as the build_od_mode_share.py options)
DATA_DIR = Path("data")
MODES = [4, 5, 6, 7, 8, 9]          # KT mode codes kept, in this order; list(range(1, 10)) for all
WEIGHT = "trips_week"               # trips_week | trips_weekday_mean | trips_weekend_mean
MIN_TRIPS = 100                     # dongs with fewer intra-dong trips are left out of the comparisons
N_CLUSTERS = 5                      # clusters of dongs with similar intra-dong mode splits

COLORS = {1: "#6c757d", 2: "#343a40", 3: "#9d4edd",
          4: "#8172b3", 5: "#2a9d8f", 6: "#4c72b0", 7: "#e9c46a", 8: "#c1121f", 9: "#adb5bd"}
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 160, "display.max_columns", 30)

## 1. Load data and build the per-dong tables

In [ ]:
def load_mode_names(path: Path) -> dict[int, str]:
    # same auto-detection as build_od_mode_share.py
    if not path.exists():
        return {}
    df = pd.read_csv(path)
    code_col = next((c for c in df.columns if pd.api.types.is_integer_dtype(df[c])), None)
    name_cols = [c for c in df.columns if c != code_col and not pd.api.types.is_numeric_dtype(df[c])]
    if code_col is None or not name_cols:
        return {}
    return {int(c): str(n) for c, n in zip(df[code_col], df[name_cols[0]])}


def find_col(df, *keys):
    return next((c for c in df.columns if all(k in c.lower() for k in keys)), None)


# ---- dongs define the index order
dongs = pd.read_csv(DATA_DIR / "nodes_dongs.csv", dtype={"dong_code": str})
codes = dongs["dong_code"].tolist()
names = dongs["name"].to_numpy()
idx = {c: i for i, c in enumerate(codes)}
N = len(codes)
gu_col = find_col(dongs, "gu", "name") or find_col(dongs, "gu")
area_col = find_col(dongs, "area")
gu = dongs[gu_col].astype(str).to_numpy() if gu_col else np.full(N, "n/a")
area = dongs[area_col].to_numpy(float) if area_col else np.full(N, np.nan)
print(f"dongs: {N}  |  gu column: {gu_col!r}  |  area column: {area_col!r}")

# ---- modes
mode_names_all = load_mode_names(DATA_DIR / "mode_codes.csv")
kidx = {c: k for k, c in enumerate(MODES)}
m = len(MODES)
mnames = [mode_names_all.get(c, f"mode {c}") for c in MODES]
mcolors = [COLORS.get(c, "#555555") for c in MODES]
print("modes:", dict(zip(MODES, mnames)))

# ---- flows
flows = pd.read_csv(DATA_DIR / "flows_od_by_mode.csv.gz", dtype={"origin": str, "destination": str})
total_all = flows[WEIGHT].sum()
intra_all = flows.loc[flows["origin"] == flows["destination"], WEIGHT].sum()
flows = flows[flows["mode"].isin(MODES)].copy()
flows["i"] = flows["origin"].map(idx)
flows["j"] = flows["destination"].map(idx)
bad = flows["i"].isna() | flows["j"].isna()
if bad.any():
    print(f"WARNING: dropping {bad.sum()} rows whose dong code is not in nodes_dongs.csv")
    flows = flows[~bad]
flows["i"] = flows["i"].astype(int)
flows["j"] = flows["j"].astype(int)
flows["k"] = flows["mode"].map(kidx).astype(int)
print(f"trips kept (modes {MODES}): {100 * flows[WEIGHT].sum() / total_all:.3f}% of all trips")
print(f"intra-dong trips: {100 * intra_all / total_all:.2f}% of all trips (all modes)")

In [ ]:
# ---- full OD x mode tensor (identical to build_od_mode_share.py); used for the outgoing split in section 5
trips = np.zeros((N, N, m))
np.add.at(trips, (flows["i"].to_numpy(), flows["j"].to_numpy(), flows["k"].to_numpy()), flows[WEIGHT].to_numpy())

# ---- per-dong intra-dong tables, shape (N, m)
intra = flows[flows["i"] == flows["j"]]
ii, kk = intra["i"].to_numpy(), intra["k"].to_numpy()


def per_dong_sum(col):
    out = np.zeros((N, m))
    np.add.at(out, (ii, kk), intra[col].to_numpy(float))
    return out


def per_dong_wmean(col, wcol=WEIGHT):
    # trip-weighted mean of a per-row mean (KT's mean_dist_m, mean_time_min)
    v, w = intra[col].to_numpy(float), intra[wcol].to_numpy(float)
    ok = np.isfinite(v) & (w > 0)
    num, den = np.zeros((N, m)), np.zeros((N, m))
    np.add.at(num, (ii[ok], kk[ok]), (w * v)[ok])
    np.add.at(den, (ii[ok], kk[ok]), w[ok])
    return np.divide(num, den, out=np.full((N, m), np.nan), where=den > 0)


def shares_of(T):
    tot = T.sum(axis=-1, keepdims=True)
    return np.divide(T, tot, out=np.full_like(T, np.nan, dtype=float), where=tot > 0)


T = per_dong_sum(WEIGHT)                       # intra-dong trips per dong and mode
assert np.allclose(T, trips[np.arange(N), np.arange(N)]), "per-dong table must equal the tensor diagonal"
P = shares_of(T)                               # intra-dong mode split per dong (rows sum to 1)
T_wd = per_dong_sum("trips_weekday_mean")      # used for the weekday/weekend comparison
T_we = per_dong_sum("trips_weekend_mean")
DIST = per_dong_wmean("mean_dist_m") / 1000    # km
TIME = per_dong_wmean("mean_time_min")         # min

n_intra = T.sum(axis=1)
n_out_all = trips.sum(axis=(1, 2))             # every trip leaving dong d (incl. intra), kept modes
self_containment = np.divide(n_intra, n_out_all, out=np.full(N, np.nan), where=n_out_all > 0)
valid = n_intra >= MIN_TRIPS
city = T.sum(axis=0) / T.sum()                 # pooled city-wide intra-dong split (what the script prints)

print(f"dongs with >= {MIN_TRIPS} intra-dong trips: {valid.sum()} of {N}")
print(f"intra-dong trips per dong: median {np.median(n_intra):,.0f}, "
      f"range {n_intra.min():,.0f} - {n_intra.max():,.0f}")

In [ ]:
# ---- one tidy table, one row per dong
per_dong = pd.DataFrame({
    "dong_code": codes, "name": names, "gu": gu, "area_km2": area,
    "intra_trips": n_intra, "all_trips_from": n_out_all, "self_containment": self_containment,
})
for k, n in enumerate(mnames):
    per_dong[f"share_{n}"] = P[:, k]
for k, n in enumerate(mnames):
    per_dong[f"dist_km_{n}"] = DIST[:, k]
for k, n in enumerate(mnames):
    per_dong[f"time_min_{n}"] = TIME[:, k]
per_dong["valid"] = valid
share_cols = [f"share_{n}" for n in mnames]
per_dong.head()

### Maps helper
Uses the `dongs` polygon layer of `seoul_multimodal_network.gpkg` if present, otherwise the representative points.

In [ ]:
GPKG = DATA_DIR / "seoul_multimodal_network.gpkg"
geo = None
if gpd is not None and GPKG.exists():
    geo = gpd.read_file(GPKG, layer="dongs")
    dc = find_col(geo, "dong", "code") or geo.columns[0]
    geo = geo.rename(columns={dc: "dong_code"})
    geo["dong_code"] = geo["dong_code"].astype(str)
    geo = geo.set_index("dong_code").reindex(codes)
    print("polygons:", geo.geometry.notna().sum(), "of", N)


def dong_map(ax, values, title, cmap="viridis", categorical=False, vmin=None, vmax=None):
    v = pd.Series(np.asarray(values), index=codes)
    if geo is not None:
        vals = v.map(lambda x: None if pd.isna(x) else str(int(x))) if categorical else v
        g = geo.assign(_v=vals.values)
        if g["_v"].isna().any():
            g[g["_v"].isna()].plot(ax=ax, color="#e9ecef", linewidth=0.1, edgecolor="white")
        g[g["_v"].notna()].plot(column="_v", ax=ax, cmap=cmap, categorical=categorical, legend=True,
                                vmin=vmin, vmax=vmax, linewidth=0.1, edgecolor="white",
                                legend_kwds={} if categorical else {"shrink": 0.6})
    else:
        ok = v.notna().values
        ax.scatter(dongs["lon"][~ok], dongs["lat"][~ok], s=8, c="#e9ecef")
        sc = ax.scatter(dongs["lon"][ok], dongs["lat"][ok], s=14, c=v[ok], cmap=cmap, vmin=vmin, vmax=vmax)
        if categorical:
            for c in np.unique(v[ok]):
                ax.scatter([], [], color=sc.cmap(sc.norm(c)), label=str(int(c)))
            ax.legend(fontsize=7, title="cluster")
        else:
            plt.colorbar(sc, ax=ax, shrink=0.6)
        ax.set_aspect(1 / np.cos(np.radians(37.55)))
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()

## 2. How different are the dongs?

Pooled city-wide intra-dong split vs. the distribution of the per-dong splits.
`mean over dongs` weights every dong equally; `pooled` weights by trips (= what `build_od_mode_share.py` prints).

In [ ]:
Pv = P[valid]
nv = names[valid]
summary = pd.DataFrame({
    "pooled %": 100 * city,
    "mean over dongs %": 100 * Pv.mean(0),
    "std %": 100 * Pv.std(0, ddof=1),
    "CV": Pv.std(0, ddof=1) / Pv.mean(0),
    "p5 %": 100 * np.percentile(Pv, 5, axis=0),
    "median %": 100 * np.median(Pv, axis=0),
    "p95 %": 100 * np.percentile(Pv, 95, axis=0),
    "min %": 100 * Pv.min(0), "dong at min": nv[Pv.argmin(0)],
    "max %": 100 * Pv.max(0), "dong at max": nv[Pv.argmax(0)],
}, index=mnames)
summary.round(2)

In [ ]:
order = np.argsort(-P[:, mnames.index(next((n for n in mnames if "walk" in n.lower()), mnames[0]))])
order = order[valid[order]]

fig, ax = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1, 2.2]})
parts = ax[0].violinplot([Pv[:, k] for k in range(m)], showmedians=True, widths=0.85)
for b, c in zip(parts["bodies"], mcolors):
    b.set_facecolor(c); b.set_alpha(0.7)
ax[0].scatter(np.arange(1, m + 1), city, marker="D", color="k", zorder=3, s=22, label="pooled city-wide")
ax[0].set_xticks(np.arange(1, m + 1), mnames, rotation=30, ha="right")
ax[0].set_ylabel("intra-dong mode share")
ax[0].set_title("Per-dong intra-dong split (one point per dong)")
ax[0].legend(fontsize=8)

bottom = np.zeros(len(order))
for k in range(m):
    ax[1].bar(np.arange(len(order)), P[order, k], bottom=bottom, color=mcolors[k], width=1.0, label=mnames[k])
    bottom += P[order, k]
ax[1].set_xlim(-0.5, len(order) - 0.5); ax[1].set_ylim(0, 1)
ax[1].set_xlabel("dongs, sorted by walk share"); ax[1].set_ylabel("share")
ax[1].set_title("Intra-dong mode split of every dong")
ax[1].legend(ncol=m, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 9))
for a, k in zip(ax.ravel(), range(m)):
    dong_map(a, np.where(valid, P[:, k], np.nan), f"intra-dong share: {mnames[k]}", cmap="viridis")
for a in ax.ravel()[m:]:
    a.set_axis_off()
plt.tight_layout(); plt.show()

## 3. Is the spread real?

**(a) Homogeneity test.** A $\chi^2$ test on the $N_\text{dongs}\times m$ table of intra-dong trips. With millions of (KT-expanded) trips the p-value is always ~0, so the useful number is the effect size, Cramér's $V$ (0 = all dongs identical, 1 = each dong uses a single, different mode).

**(b) Overdispersion.** If every dong had the city-wide split $p_k$ and the differences were pure multinomial sampling noise, the share of mode $k$ in dong $d$ would have variance $p_k(1-p_k)/n_d$.
The ratio $\phi_k = \mathrm{Var}_d(P[d,k]) \,/\, \overline{p_k(1-p_k)/n_d}$ says how many times larger the observed spread is.
Since KT counts are expanded, the true sample is smaller than $n_d$; $n^*_k = p_k(1-p_k)/\mathrm{Var}_d(P[d,k])$ is the number of trips per dong at which noise alone *would* explain the spread. If $n^*$ is tiny compared with the observed counts, the spread is not sampling noise.

**(c) Distance to the city-wide split.** Per dong: Jensen–Shannon distance (base 2, in $[0,1]$) and total variation distance $\mathrm{TVD}=\tfrac12\sum_k|P[d,k]-p_k|$, i.e. the fraction of the dong's intra trips that would have to switch mode to match the city split.
As a yardstick we compute the same distances between each dong's **weekday and weekend** split: if differences *between* dongs are much larger than the weekday/weekend change *within* a dong, they are structural.

In [ ]:
# (a) chi-square homogeneity + Cramer's V
Tv = T[valid]
Tv_nz = Tv[:, Tv.sum(0) > 0]
chi2, pval, dof, _ = stats.chi2_contingency(Tv_nz)
cramers_v = np.sqrt(chi2 / (Tv_nz.sum() * (min(Tv_nz.shape) - 1)))
print(f"chi2 = {chi2:.3e}, dof = {dof}, p = {pval:.1e}   ->  Cramer's V = {cramers_v:.3f}")

# (b) overdispersion per mode
nvd = n_intra[valid]
var_obs = Pv.var(0, ddof=1)
var_noise = (city * (1 - city))[None, :] / nvd[:, None]
overdisp = pd.DataFrame({
    "observed std (pp)": 100 * np.sqrt(var_obs),
    "noise std (pp)": 100 * np.sqrt(var_noise.mean(0)),
    "phi = var_obs / var_noise": var_obs / var_noise.mean(0),
    "n* (trips/dong)": city * (1 - city) / var_obs,
}, index=mnames)
print(f"\nmedian observed intra-dong trips per dong: {np.median(nvd):,.0f}")
overdisp.round(3)

In [ ]:
def js_dist(P, Q):
    # Jensen-Shannon distance (base 2) between rows of P and Q (broadcastable), in [0, 1]
    P, Q = np.broadcast_arrays(np.asarray(P, float), np.asarray(Q, float))
    M = 0.5 * (P + Q)
    with np.errstate(divide="ignore", invalid="ignore"):
        kl_pm = np.where(P > 0, P * np.log2(P / M), 0.0).sum(-1)
        kl_qm = np.where(Q > 0, Q * np.log2(Q / M), 0.0).sum(-1)
    return np.sqrt(np.clip(0.5 * (kl_pm + kl_qm), 0, None))


def tvd(P, Q):
    return 0.5 * np.abs(np.asarray(P) - np.asarray(Q)).sum(-1)


P_wd, P_we = shares_of(T_wd), shares_of(T_we)
ok_t = valid & (T_wd.sum(1) > 0) & (T_we.sum(1) > 0)

per_dong["js_vs_city"] = js_dist(P, city)
per_dong["tvd_vs_city"] = tvd(P, city)
per_dong["js_weekday_weekend"] = np.where(ok_t, js_dist(P_wd, P_we), np.nan)
per_dong["tvd_weekday_weekend"] = np.where(ok_t, tvd(P_wd, P_we), np.nan)

# pairwise distances between all valid dongs
D_js = js_dist(Pv[:, None, :], Pv[None, :, :])
iu = np.triu_indices(len(Pv), 1)

comp = pd.DataFrame({
    "dong vs city-wide": per_dong.loc[valid, "tvd_vs_city"].describe(percentiles=[.1, .5, .9]),
    "dong vs dong (pairs)": pd.Series(tvd(Pv[:, None, :], Pv[None, :, :])[iu]).describe(percentiles=[.1, .5, .9]),
    "weekday vs weekend (same dong)": per_dong.loc[ok_t, "tvd_weekday_weekend"].describe(percentiles=[.1, .5, .9]),
}).T[["count", "mean", "10%", "50%", "90%", "max"]]
print("Total variation distance between intra-dong mode splits")
comp.round(3)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
bins = np.linspace(0, np.nanmax([per_dong["js_vs_city"].max(), D_js[iu].max()]) * 1.02, 50)
ax[0].hist(D_js[iu], bins=bins, density=True, alpha=0.5, color="slategray", label="dong vs dong")
ax[0].hist(per_dong.loc[valid, "js_vs_city"], bins=bins, density=True, alpha=0.6, color="#2a9d8f", label="dong vs city-wide")
ax[0].hist(per_dong.loc[ok_t, "js_weekday_weekend"], bins=bins, density=True, alpha=0.6, color="#e76f51",
           label="weekday vs weekend, same dong")
ax[0].set_xlabel("Jensen-Shannon distance"); ax[0].set_ylabel("density"); ax[0].legend(fontsize=8)
ax[0].set_title("Between-dong vs within-dong (temporal) differences")

ax[1].scatter(per_dong.loc[ok_t, "js_weekday_weekend"], per_dong.loc[ok_t, "js_vs_city"], s=10, alpha=0.6)
lim = np.nanmax(per_dong.loc[ok_t, ["js_weekday_weekend", "js_vs_city"]].to_numpy()) * 1.05
ax[1].plot([0, lim], [0, lim], "k--", lw=0.8, label="y = x")
ax[1].set_xlabel("JS weekday vs weekend (same dong)"); ax[1].set_ylabel("JS dong vs city-wide")
ax[1].set_title("Points above y=x: dong differs from the city\nmore than from itself on weekends", fontsize=9)
ax[1].legend(fontsize=8)

dong_map(ax[2], np.where(valid, per_dong["js_vs_city"], np.nan), "JS distance of intra-dong split from city-wide",
         cmap="magma_r")
plt.tight_layout(); plt.show()

frac_above = (per_dong.loc[ok_t, "js_vs_city"] > per_dong.loc[ok_t, "js_weekday_weekend"]).mean()
print(f"{100 * frac_above:.1f}% of dongs differ more from the city-wide split than from their own weekend split")

In [ ]:
# the most atypical dongs, with their split
atyp = per_dong[valid].sort_values("js_vs_city", ascending=False)
show = atyp[["name", "gu", "intra_trips", "js_vs_city", "tvd_vs_city"] + share_cols].head(15).copy()
show[share_cols] = (100 * show[share_cols]).round(1)
print("city-wide %:", {n: round(float(v), 1) for n, v in zip(mnames, 100 * city)})
show.round(3)

## 4. What do the differences follow?

**(a) Gu.** Fraction of the between-dong variance of each share explained by the gu ($\eta^2$ of a one-way ANOVA) plus a Kruskal–Wallis test.
To judge $\eta^2$, compare it with a permutation null where dongs are shuffled across gus (same gu sizes).

**(b) Dong covariates.** Spearman correlations of the shares with dong area, intra-dong volume, self-containment (fraction of the dong's trips that stay inside), and mean intra-dong trip length.

In [ ]:
gv = gu[valid]
groups = pd.Series(gv).groupby(gv).indices
rng = np.random.default_rng(0)


def eta2(x, labels):
    grand = x.mean()
    ss_b = sum(len(ix) * (x[ix].mean() - grand) ** 2 for ix in pd.Series(labels).groupby(labels).indices.values())
    return ss_b / ((x - grand) ** 2).sum()


rows = []
for k, n in enumerate(mnames):
    x = Pv[:, k]
    e = eta2(x, gv)
    null = np.array([eta2(x, rng.permutation(gv)) for _ in range(500)])
    h, p = stats.kruskal(*[x[ix] for ix in groups.values()]) if len(groups) > 1 else (np.nan, np.nan)
    rows.append({"mode": n, "eta2 (gu)": e, "eta2 null mean": null.mean(), "eta2 null p95": np.percentile(null, 95),
                 "perm p": (null >= e).mean(), "Kruskal-Wallis H": h, "KW p": p})
gu_tab = pd.DataFrame(rows).set_index("mode")
print(f"{len(groups)} gus, {valid.sum()} dongs")
gu_tab.round(4)

In [ ]:
gu_mean = per_dong[valid].groupby("gu")[share_cols].mean()
gu_mean = gu_mean.sort_values(share_cols[mnames.index(next((n for n in mnames if "walk" in n.lower()), mnames[0]))])
fig, ax = plt.subplots(figsize=(14, 4.5))
bottom = np.zeros(len(gu_mean))
for k in range(m):
    ax.bar(gu_mean.index, gu_mean[share_cols[k]], bottom=bottom, color=mcolors[k], label=mnames[k])
    bottom += gu_mean[share_cols[k]].to_numpy()
ax.set_ylim(0, 1); ax.set_ylabel("mean intra-dong share over the gu's dongs")
ax.tick_params(axis="x", rotation=60)
ax.legend(ncol=m, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.32))
ax.set_title("Intra-dong mode split averaged by gu")
plt.tight_layout(); plt.show()

In [ ]:
walk_k = next((k for k, n in enumerate(mnames) if "walk" in n.lower()), None)
cov = pd.DataFrame({
    "log area": np.log(area),
    "log intra trips": np.log(n_intra.clip(1)),
    "self-containment": self_containment,
    "mean intra trip km (all modes)": np.nansum(np.nan_to_num(DIST) * T, 1) / np.where(n_intra > 0, n_intra, np.nan),
})
if walk_k is not None:
    cov["mean intra walk km"] = DIST[:, walk_k]
cov = cov[valid]
rho = pd.DataFrame({c: [stats.spearmanr(cov[c], Pv[:, k], nan_policy="omit")[0] for k in range(m)]
                    for c in cov.columns}, index=mnames)

fig, ax = plt.subplots(figsize=(1.4 * rho.shape[1] + 2, 0.5 * m + 1.5))
im = ax.imshow(rho.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(rho.shape[1]), rho.columns, rotation=30, ha="right"); ax.set_yticks(range(m), mnames)
for (a, b), v in np.ndenumerate(rho.to_numpy()):
    ax.text(b, a, f"{v:.2f}", ha="center", va="center", fontsize=8, color="w" if abs(v) > 0.5 else "k")
plt.colorbar(im, ax=ax, label="Spearman rho"); ax.set_title("Intra-dong mode share vs dong covariates")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
x = np.sqrt(area[valid])
for k in range(m):
    ax[0].scatter(x, DIST[valid, k], s=6, alpha=0.5, color=mcolors[k], label=mnames[k])
ax[0].set_xlabel("sqrt(dong area) [km]"); ax[0].set_ylabel("mean intra-dong trip distance [km]")
ax[0].set_title("Intra-dong trip length vs dong size"); ax[0].legend(fontsize=7, markerscale=2)

ax[1].boxplot([DIST[valid, k][np.isfinite(DIST[valid, k])] for k in range(m)], showfliers=False)
ax[1].set_xticks(np.arange(1, m + 1), mnames, rotation=30, ha="right")
ax[1].set_ylabel("km"); ax[1].set_title("Mean intra-dong distance, across dongs")
ax[2].boxplot([TIME[valid, k][np.isfinite(TIME[valid, k])] for k in range(m)], showfliers=False)
ax[2].set_xticks(np.arange(1, m + 1), mnames, rotation=30, ha="right")
ax[2].set_ylabel("min"); ax[2].set_title("Mean intra-dong travel time, across dongs")
plt.tight_layout(); plt.show()

### (c) Clusters of dongs with similar intra-dong splits

Ward clustering on the Hellinger embedding $\sqrt{P[d,\cdot]}/\sqrt{2}$ (Euclidean distance there is the Hellinger distance between splits, so Ward is well defined).

In [ ]:
H = np.sqrt(Pv) / np.sqrt(2)
Z = linkage(H, method="ward")
lab_v = fcluster(Z, N_CLUSTERS, criterion="maxclust")
cluster = np.full(N, np.nan)
cluster[valid] = lab_v
per_dong["cluster"] = cluster

prof = pd.DataFrame(Pv, columns=mnames).groupby(lab_v).mean()
prof["n dongs"] = np.bincount(lab_v)[1:]
prof["intra trips (median)"] = pd.Series(nvd).groupby(lab_v).median()
prof["area km2 (median)"] = pd.Series(area[valid]).groupby(lab_v).median()
prof["examples"] = pd.Series(nv).groupby(lab_v).apply(lambda s: ", ".join(s.iloc[:4]))
display_prof = prof.copy()
display_prof[mnames] = (100 * display_prof[mnames]).round(1)
display_prof

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8), gridspec_kw={"width_ratios": [1.1, 0.9, 1.2]})
dendrogram(Z, ax=ax[0], no_labels=True, color_threshold=Z[-(N_CLUSTERS - 1), 2], above_threshold_color="#999")
ax[0].set_title(f"Ward dendrogram ({N_CLUSTERS} clusters)"); ax[0].set_ylabel("Ward distance")

bottom = np.zeros(N_CLUSTERS)
for k in range(m):
    ax[1].bar(prof.index.astype(str), prof[mnames[k]], bottom=bottom, color=mcolors[k], label=mnames[k])
    bottom += prof[mnames[k]].to_numpy()
for c, n in zip(prof.index, prof["n dongs"]):
    ax[1].text(str(c), 1.01, f"n={n}", ha="center", fontsize=8)
ax[1].set_ylim(0, 1.08); ax[1].set_xlabel("cluster"); ax[1].set_ylabel("mean share")
ax[1].set_title("Cluster profiles"); ax[1].legend(fontsize=7, loc="lower right")

dong_map(ax[2], cluster, "Clusters of intra-dong mode split", cmap="tab10", categorical=True)
plt.tight_layout(); plt.show()

## 5. Intra-dong vs outgoing split

For each dong, compare the split of its intra-dong trips ($S[d,d,\cdot]$) with the split of the trips **leaving** it to other dongs ($\sum_{j\neq d}$ trips$[d,j,\cdot]$, normalised).
This also tells whether dongs differ more in how they move internally or in how they connect to the rest of the city.

In [ ]:
out_T = trips.sum(axis=1) - T                      # inter-dong trips leaving d, per mode
P_out = shares_of(out_T)
ok_o = valid & (out_T.sum(1) > 0)
city_out = out_T.sum(0) / out_T.sum()

per_dong["js_intra_vs_out"] = np.where(ok_o, js_dist(P, P_out), np.nan)
per_dong["js_out_vs_city_out"] = np.where(ok_o, js_dist(P_out, city_out), np.nan)

cmp = pd.DataFrame({
    "intra: pooled %": 100 * city, "outgoing: pooled %": 100 * city_out,
    "intra: std across dongs (pp)": 100 * P[ok_o].std(0, ddof=1),
    "outgoing: std across dongs (pp)": 100 * P_out[ok_o].std(0, ddof=1),
    "Spearman(intra, outgoing) across dongs": [stats.spearmanr(P[ok_o, k], P_out[ok_o, k])[0] for k in range(m)],
}, index=mnames)
print(f"median JS(dong, city)  intra: {per_dong.loc[ok_o, 'js_vs_city'].median():.3f}   "
      f"outgoing: {per_dong.loc[ok_o, 'js_out_vs_city_out'].median():.3f}")
print(f"median JS(intra split, outgoing split) of the same dong: {per_dong.loc[ok_o, 'js_intra_vs_out'].median():.3f}")
cmp.round(3)

In [ ]:
fig, ax = plt.subplots(1, m, figsize=(2.9 * m, 3.0), sharex=False)
for k in range(m):
    a = ax[k]
    a.scatter(P_out[ok_o, k], P[ok_o, k], s=6, alpha=0.5, color=mcolors[k])
    hi = np.nanmax([P_out[ok_o, k].max(), P[ok_o, k].max()]) * 1.05
    a.plot([0, hi], [0, hi], "k--", lw=0.7)
    a.set_title(mnames[k]); a.set_xlabel("outgoing share")
    if k == 0:
        a.set_ylabel("intra-dong share")
plt.suptitle("Per dong: intra-dong vs outgoing share (dashed: equal)", y=1.03)
plt.tight_layout(); plt.show()

## 6. Summary (computed)

In [ ]:
walk_k = next((k for k, n in enumerate(mnames) if "walk" in n.lower()), 0)
car_k = next((k for k, n in enumerate(mnames) if "car" in n.lower()), None)
med_tvd_city = per_dong.loc[valid, "tvd_vs_city"].median()
med_tvd_we = per_dong.loc[ok_t, "tvd_weekday_weekend"].median()
top_gu = gu_tab["eta2 (gu)"].idxmax()

print(f"* {valid.sum()} dongs compared (>= {MIN_TRIPS} intra-dong trips), weight = {WEIGHT}, modes = {mnames}")
print(f"* Cramer's V of the dong x mode table: {cramers_v:.3f}")
print(f"* {mnames[walk_k]} share of intra-dong trips: pooled {100 * city[walk_k]:.1f}%, "
      f"dongs range p5-p95 {100 * np.percentile(Pv[:, walk_k], 5):.1f}-{100 * np.percentile(Pv[:, walk_k], 95):.1f}%")
if car_k is not None:
    print(f"* {mnames[car_k]} share of intra-dong trips: pooled {100 * city[car_k]:.1f}%, "
          f"dongs range p5-p95 {100 * np.percentile(Pv[:, car_k], 5):.1f}-{100 * np.percentile(Pv[:, car_k], 95):.1f}%")
print(f"* spread vs multinomial noise: phi between {overdisp.iloc[:, 2].min():.3g} and {overdisp.iloc[:, 2].max():.3g}; "
      f"noise alone would need only {overdisp.iloc[:, 3].min():.0f}-{overdisp.iloc[:, 3].max():.0f} trips per dong "
      f"(observed median {np.median(nvd):,.0f})")
print(f"* median TVD dong vs city: {100 * med_tvd_city:.1f}% of intra trips would need to change mode; "
      f"weekday vs weekend of the same dong: {100 * med_tvd_we:.1f}%"
      + (f"  (ratio {med_tvd_city / med_tvd_we:.1f}x)" if med_tvd_we > 1e-6 else ""))
print(f"* gu explains most for {top_gu}: eta2 = {gu_tab.loc[top_gu, 'eta2 (gu)']:.2f} "
      f"(permutation null p95 = {gu_tab.loc[top_gu, 'eta2 null p95']:.2f})")
print(f"* strongest covariate per mode: "
      + "; ".join(f"{n}: {rho.loc[n].abs().idxmax()} (rho={rho.loc[n, rho.loc[n].abs().idxmax()]:+.2f})" for n in mnames))

In [ ]:
# optional: save the per-dong table
# per_dong.to_csv("intra_dong_mode_share_by_dong.csv", index=False)